# Learning Objectives

* Build a hydrologic modeling workflow (NextGen / NGIAB) that scatters over USGS gages.
* Generate the workflow's catalogs and DAG with the same generator class the command-line tool uses.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the simulated hydrograph and the evaluation metrics against USGS observations and NWM v3.0.

# NextGen (NGIAB) Watershed Modeling Workflow

This workflow reproduces the five-notebook NextGen In A Box pipeline of Nassar et al. 2026 (*Environmental
Modelling & Software* 203, 107031) as a Pegasus DAG: subset the CONUS hydrofabric at a gage, regrid AORC forcings,
run NextGen (NOAH-OWP + CFE + t-route), and evaluate the streamflow against USGS observations (RMSE/KGE) and the NWM
v3.0 retrospective (TEEHR). An optional SPOTPY DDS calibration branch is off in this notebook. See `README.md` for
the full pipeline and options.

**Workflow Jobs (per gage, after one shared hydrofabric fetch):**
1. `fetch_hydrofabric` - download the CONUS hydrofabric once, shared by all gages
2. `subset_hydrofabric`, `generate_forcings`, `generate_realization` - prepare the basin
3. `assemble_rundir` - merge them into one NGIAB run directory
4. `run_nextgen` - NOAH-OWP + CFE + t-route
5. `fetch_usgs_obs`, `outputs_analysis` - observations, basin means, RMSE/KGE
6. `teehr_evaluation` - TEEHR evaluation against the NWM v3.0 retrospective
7. `summarize` - one table and plot across all gages

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment. Workers need ~14 GB of memory and internet access (hydrofabric, AORC, USGS, NWM).

## 1. Choose the Basin and Period

The data is fetched at run time by the workflow's own jobs. This notebook runs the paper's demo basin, USGS gage
10109001, over water years 2018-2021, with calibration off — the smoke configuration, about 40
minutes on a small HTCondor pool. Add gages, or set `CALIBRATE` to the number of DDS repetitions (the paper uses 6;
each is a full model run), to scale up.

In [ ]:
GAGES = ["gage-10109001"]
START, END = "2017-10-01", "2021-09-30"
CALIBRATE = 0          # DDS repetitions; 0 disables the calibration branch
HYDROFABRIC_TAR = None # a pre-downloaded hydrofabric_cache.tar skips the multi-GB fetch

## 2. Build the Apptainer Containers

The model jobs run in `Apptainer/NextGen_Container.def` (CIROH's NGIAB engine image plus the wrappers' Python
dependencies); `teehr_evaluation` runs in `Apptainer/Teehr_Container.def` (CIROH's TEEHR image). Both are large;
skip the rebuild if the `.sif` files already exist.

In [ ]:
%%bash
for name in NextGen Teehr; do
    if [ ! -f Apptainer/${name}_Container.sif ]; then
        apptainer build Apptainer/${name}_Container.sif Apptainer/${name}_Container.def
    else
        echo "Container already built: Apptainer/${name}_Container.sif"
    fi
done

## 3. Generate the Workflow

The following cell uses the `NextGenWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and writes it to
`workflow.yml`. The workflow-specific inputs are the same values the CLI's `--gages`/`--start`/`--end`/`--calibrate`
options set.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import NextGenWorkflow

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The inputs the CLI would parse
ARGS = argparse.Namespace(gages=GAGES, start=START, end=END, training_start=None,
                          calibrate=CALIBRATE, dds_trials=1,
                          hydrofabric_tar=HYDROFABRIC_TAR)

workflow = NextGenWorkflow(dagfile=DAGFILE)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.calibrate_repetitions = CALIBRATE
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog(hydrofabric_tar=HYDROFABRIC_TAR)

print("Creating NextGen workflow DAG...")
workflow.create_workflow(ARGS)

workflow.write()
print("\nNextGen Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The shared hydrofabric fetch feeds each
gage's preparation chain; the model run fans out to the analysis and TEEHR jobs, and `summarize` joins all gages.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog chosen
above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and prints
the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/` (to `wf-output/` instead when
`HOSTED_SITE_CATALOG` is set: the hosted catalog has no `local` site, so Pegasus uses its default one), per gage:

| Output | Description |
|--------|-------------|
| `{gage}_usgs_obs.csv` | USGS observed streamflow |
| `analysis/{gage}_basin_means.csv` | Basin-mean model outputs |
| `analysis/{gage}_metrics.csv` | RMSE / KGE against USGS observations |
| `analysis/{gage}_teehr_metrics.csv` | TEEHR metrics against USGS and NWM v3.0 |
| `plots/{gage}_hydrograph.png`, `plots/{gage}_water_balance.png`, `plots/{gage}_teehr.png` | Plots |
| `summary/summary_metrics.csv`, `summary/summary_metrics.png` | All gages together |

In [ ]:
!ls -lR output/

The simulated and observed hydrograph, and the model's skill against USGS observations:

In [ ]:
import pandas as pd

GAGE = GAGES[0]
display(Image(filename=f"output/plots/{GAGE}_hydrograph.png"))
pd.read_csv(f"output/analysis/{GAGE}_metrics.csv")